# Mask Validation: Multi-Otsu Thresholding

Validates the auto-generated masks from Multi-Otsu thresholding.
Examines fallback frequency, threshold distribution, and mask quality.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import yaml
from pathlib import Path
import sys

sys.path.insert(0, str(Path('..').resolve()))
from src.data.mask_generator import generate_mask, clean_mask, validate_mask_quality

plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150

In [ ]:
with open('../configs/config.yaml') as f:
    config = yaml.safe_load(f)

patches_dir = Path(config['data']['base_dir']) / 'patches'
mixed_train = np.load(patches_dir / 'mixed' / 'train.npy')
print(f"Loaded {len(mixed_train)} mixed training patches")

## 1. Mask Generation Visualization

In [ ]:
n_samples = 10
indices = np.random.choice(len(mixed_train), n_samples, replace=False)

fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4*n_samples))

methods_used = []
for i, idx in enumerate(indices):
    patch = mixed_train[idx]
    mask, method = generate_mask(patch, 'mixed')
    cleaned = clean_mask(mask)
    methods_used.append(method)
    
    axes[i, 0].imshow(patch, cmap='gray', vmin=0, vmax=1)
    axes[i, 0].set_title(f'Raw Patch (std={patch.std():.4f})')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(mask, cmap='gray', vmin=0, vmax=1)
    axes[i, 1].set_title(f'Raw Mask ({method})')
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(patch, cmap='gray', vmin=0, vmax=1)
    axes[i, 2].imshow(cleaned, cmap='Reds', alpha=0.4)
    axes[i, 2].set_title('Cleaned Overlay')
    axes[i, 2].axis('off')

plt.suptitle('Mask Generation Validation', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Fallback Rate Analysis

In [ ]:
batch_size = min(500, len(mixed_train))
indices = np.random.choice(len(mixed_train), batch_size, replace=False)
patches = mixed_train[indices]

methods = []
for p in patches:
    _, method = generate_mask(p, 'mixed')
    methods.append(method)

stats = validate_mask_quality(patches, np.zeros_like(patches), methods)

print(f"Total patches analyzed: {stats['total_patches']}")
print(f"Method distribution: {stats['method_counts']}")
print(f"Fallback rate: {stats['fallback_rate']:.2%}")
print(f"Warning (fallback > 20%): {stats['warning_low_std']}")

## 3. Threshold Distribution

In [ ]:
from skimage.filters import threshold_multiotsu

thresholds_list = []
for p in patches[:200]:
    try:
        t = threshold_multiotsu(p, classes=3)
        thresholds_list.append(t[0])
    except ValueError:
        pass

if thresholds_list:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(thresholds_list, bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(np.mean(thresholds_list), color='r', linestyle='--', 
               label=f'Mean: {np.mean(thresholds_list):.4f}')
    ax.set_xlabel('First Otsu Threshold')
    ax.set_ylabel('Count')
    ax.set_title('Multi-Otsu Threshold Distribution (Mixed Patches)')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No valid thresholds computed (all patches triggered fallback)")